# BEP-EpiPredict: Quick Start Tutorial

This notebook demonstrates the full prediction and interpretation workflow.
Run after training (Stage 2 + 3) or with the provided toy checkpoint.

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['pdf.fonttype'] = 42

sys.path.insert(0, str(Path('.').absolute()))

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

MARKS = cfg['histone_marks']
BEPS  = [b['name'] for b in cfg['beps'] if b['role'] != 'control']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Trained Model

In [ ]:
from models.model import BEPPerturbationModel

model = BEPPerturbationModel(cfg, use_pretrained_backbone=False)
ckpt  = torch.load('checkpoints/stage3_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model = model.eval().to(DEVICE)

counts = model.get_param_count()
print(f"Parameters: {counts['total']:,} total, {counts['trainable']:,} trainable")

## 2. Sensitivity Rankings — Which Mark is Most Responsive?

In [ ]:
with open('outputs/results/stage2/sensitivity_rankings.json') as f:
    rankings = json.load(f)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

colors_role = {'repressor': '#4878CF', 'activator': '#D65F5F'}
bep_meta = {b['name']: b for b in cfg['beps']}

for ax, bep in zip(axes, BEPS):
    ranked = rankings[bep]
    marks_r = [d['mark'] for d in ranked]
    si_vals = [d['SI'] for d in ranked]
    role    = bep_meta[bep]['role']
    color   = colors_role.get(role, '#888888')
    
    bars = ax.barh(range(len(marks_r))[::-1], si_vals,
                   color=color, alpha=0.8, height=0.7)
    ax.set(yticks=range(len(marks_r))[::-1],
           yticklabels=marks_r, xlabel='Sensitivity Index',
           title=bep.split('_',1)[1])
    ax.tick_params(axis='y', labelsize=7)
    ax.axvline(0, color='black', lw=0.5)

for ax in axes[len(BEPS):]:
    ax.set_visible(False)

fig.suptitle('Histone mark sensitivity per BEP\n(SI = mean|log₂FC| × fraction responsive loci)',
             fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig('outputs/results/figures/tutorial_sensitivity.pdf', bbox_inches='tight')
plt.show()

## 3. Modality Importance — What Drives BEP Effects?

In [ ]:
with open('outputs/results/stage2/shap_importance.json') as f:
    shap = json.load(f)

mark_vals  = [shap['histone_marks'].get(m, 0) for m in MARKS]
extra_vals = [shap['atac'], shap['methylation']]
extra_labs = ['ATAC-seq', 'CG Meth']
all_vals   = mark_vals + extra_vals
all_labels = MARKS + extra_labs
colors     = ['#4878CF'] * len(MARKS) + ['#D65F5F', '#6ACC65']

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(all_labels)), all_vals, color=colors)
ax.set(xticks=range(len(all_labels)),
       ylabel='|Gradient × Input|',
       title='Feature importance: which baseline signal predicts BEP effect?')
ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=8)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4878CF', label='Histone marks'),
    Patch(color='#D65F5F', label='ATAC-seq'),
    Patch(color='#6ACC65', label='CG methylation'),
])
fig.tight_layout()
plt.savefig('outputs/results/figures/tutorial_importance.pdf', bbox_inches='tight')
plt.show()

## 4. Dose-Response: dCas9 Intensity vs Mark Response

In [ ]:
with open('outputs/results/stage2/dose_response.json') as f:
    dose_data = json.load(f)

# Show top 3 BEPs with highest dynamic range
BEP_SHOW = BEPS[:4]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
axes = axes.flatten()

for ax, bep in zip(axes, BEP_SHOW):
    doses    = np.array(dose_data[bep]['doses_log2'])
    h_curves = np.array(dose_data[bep]['hist_log2fc'])  # (n_doses, n_marks)
    
    # Top 5 most dynamic marks
    amp  = h_curves.max(0) - h_curves.min(0)
    top5 = np.argsort(amp)[-5:][::-1]
    
    cmap = plt.cm.get_cmap('tab10')
    for k, j in enumerate(top5):
        ax.plot(doses, h_curves[:, j], label=MARKS[j],
                color=cmap(k), lw=2)
    
    ax.axhline(0, ls='--', color='gray', lw=0.8)
    ax.axvline(0, ls=':', color='gray', lw=0.8, alpha=0.5)
    ax.set(xlabel='log₂(dCas9 intensity scale factor)',
           ylabel='Predicted log₂FC',
           title=bep.split('_',1)[1])
    ax.legend(fontsize=7, ncol=2)

fig.suptitle('Dose-response: dCas9 binding intensity vs histone mark change\n(top 5 dynamic marks per BEP)',
             fontsize=12)
fig.tight_layout()
plt.savefig('outputs/results/figures/tutorial_dose_response.pdf', bbox_inches='tight')
plt.show()

## 5. BEP Similarity — Which BEPs Have Similar Mechanisms?

In [ ]:
with open('outputs/results/stage2/bep_similarity.json') as f:
    sim_data = json.load(f)

bep_names = sim_data['bep_names']
sim_mat   = np.array(sim_data['cosine_similarity'])
short     = [b.split('_',1)[1] for b in bep_names]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_mat, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity of predicted effect profiles')
ax.set(xticks=range(len(short)), yticks=range(len(short)),
       title='BEP effect similarity (predicted histone log₂FC vectors)')
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short, fontsize=9)

# Annotate values
for i in range(len(short)):
    for j in range(len(short)):
        ax.text(j, i, f'{sim_mat[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='black' if abs(sim_mat[i,j]) < 0.7 else 'white')

fig.tight_layout()
plt.savefig('outputs/results/figures/tutorial_bep_similarity.pdf', bbox_inches='tight')
plt.show()

## 6. K562 Transfer — Load Predictions

In [ ]:
pred_path = 'outputs/K562_all_bep_predictions.tsv'
if Path(pred_path).exists():
    df = pd.read_csv(pred_path, sep='\t')
    print(f'Predictions shape: {df.shape}')
    print(f'BEPs predicted: {df["BEP"].unique()}')
    print(f'\nSample predictions (ZIM3, top loci by |H3K27me3 log2FC|):')
    sub = df[df['BEP']=='BEP100_ZIM3'].copy()
    sub['abs_H3K27me3'] = sub['log2fc_H3K27me3'].abs()
    display(sub.nlargest(5, 'abs_H3K27me3')[['chrom','center','BEP','log2fc_H3K27me3','cls_H3K27me3','atac_log2fc','rna_log2fc']])
else:
    print(f'No prediction file found at {pred_path}')
    print('Run: python scripts/03_predict.py --ckpt checkpoints/stage3_best.pt --all_beps')

## 7. Compare H3K27me3 Distribution: Repressors vs Activators

In [ ]:
pred_path = 'outputs/K562_all_bep_predictions.tsv'
if Path(pred_path).exists():
    df = pd.read_csv(pred_path, sep='\t')
    bep_meta = {b['name']: b for b in cfg['beps']}
    df['role'] = df['BEP'].map(lambda x: bep_meta.get(x, {}).get('role', 'unknown'))

    marks_show = ['H3K27me3', 'H3K4me3', 'H3K27ac', 'H3K9me3']
    fig, axes = plt.subplots(1, len(marks_show), figsize=(14, 4))

    for ax, mark in zip(axes, marks_show):
        col = f'log2fc_{mark}'
        if col not in df.columns:
            continue
        for role, color in [('repressor', '#4878CF'), ('activator', '#D65F5F')]:
            vals = df[df['role'] == role][col].dropna()
            ax.hist(vals, bins=50, alpha=0.6, color=color, label=role, density=True)
        ax.axvline(0, ls='--', color='black', lw=0.8)
        ax.set(xlabel='Predicted log₂FC', title=mark)
        ax.legend(fontsize=8)

    fig.suptitle('K562: Predicted histone mark changes (repressors vs activators)',
                 fontsize=12)
    fig.tight_layout()
    plt.savefig('outputs/results/figures/tutorial_K562_distribution.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Run predictions first (cell 6)')